# Notebook 01 — Static Map
Phase 1.1 data: single run, 20260506
Deadzone boundary confirmed: ±130 PWM
Data file: data/raw/static_sweep_20260506_run1.csv
This is your audit trail.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/raw/static_sweep_20260506_run1.csv')

print('First 5 rows:')
display(df.head())

print('Last 5 rows:')
display(df.tail())

In [ ]:
fwd = df[df['direction'] == 'fwd'].copy()
rev = df[df['direction'] == 'rev'].copy()

print(f'Forward rows: {len(fwd)}')
print(f'Reverse rows: {len(rev)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(fwd['pwm_cmd'], fwd['vmean_v'], 'o-', color='blue', label='Forward')
ax.plot(rev['pwm_cmd'], rev['vmean_v'], 'o-', color='red', label='Reverse')

ax.axhline(0, color='grey', linestyle='--', linewidth=1)
ax.axvline(130, color='grey', linestyle='--', linewidth=1)
ax.set_title('Static Map: PWM Command vs Motor Voltage (V_mean)')
ax.set_xlabel('PWM command')
ax.set_ylabel('V_mean (V)')
ax.set_xlim(0, 255)
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('../figures/01_pwm_vs_vmean.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(fwd['pwm_cmd'], fwd['rpm'], 'o-', color='blue', label='Forward')
ax.plot(rev['pwm_cmd'], rev['rpm'], 'o-', color='red', label='Reverse')

ax.axhline(0, color='grey', linestyle='--', linewidth=1)
ax.axvline(130, color='grey', linestyle='--', linewidth=1)
ax.set_title('Static Map: PWM Command vs Steady-State RPM')
ax.set_xlabel('PWM command')
ax.set_ylabel('RPM')
ax.set_xlim(0, 255)
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('../figures/02_pwm_vs_rpm.png', dpi=150)
plt.show()

In [ ]:
fwd_moving = fwd[fwd['rpm'] != 0.0].copy()
rev_moving = rev[rev['rpm'] != 0.0].copy()

K_fwd = np.sum(fwd_moving['vmean_v'] * fwd_moving['rpm']) / np.sum(fwd_moving['vmean_v'] ** 2)
K_rev = np.sum(rev_moving['vmean_v'] * rev_moving['rpm']) / np.sum(rev_moving['vmean_v'] ** 2)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(fwd_moving['vmean_v'], fwd_moving['rpm'], 'o', color='blue', label='Forward')
ax.plot(rev_moving['vmean_v'], rev_moving['rpm'], 'o', color='red', label='Reverse')

fwd_x = np.linspace(0, fwd_moving['vmean_v'].max(), 100)
rev_x = np.linspace(rev_moving['vmean_v'].min(), 0, 100)
ax.plot(fwd_x, K_fwd * fwd_x, '-', color='blue', alpha=0.7, label=f'Forward fit: K={K_fwd:.2f} rpm/V')
ax.plot(rev_x, K_rev * rev_x, '-', color='red', alpha=0.7, label=f'Reverse fit: K={K_rev:.2f} rpm/V')

ax.axhline(0, color='grey', linestyle='--', linewidth=1)
ax.axvline(0, color='grey', linestyle='--', linewidth=1)
ax.set_title('V_mean vs Steady-State RPM (Dynamic Block Characteristic)')
ax.set_xlabel('V_mean (V)')
ax.set_ylabel('RPM')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('../figures/03_vmean_vs_rpm.png', dpi=150)
plt.show()

print(f'K_fwd = {K_fwd:.4f} rpm/V')
print(f'K_rev = {K_rev:.4f} rpm/V')

In [ ]:
fwd_move = fwd[fwd['pwm_cmd'] >= 130][['pwm_cmd', 'vmean_v', 'rpm']].copy()
rev_move = rev[rev['pwm_cmd'] >= 130][['pwm_cmd', 'vmean_v', 'rpm']].copy()

paired = fwd_move.merge(rev_move, on='pwm_cmd', suffixes=('_fwd', '_rev'))
paired['voltage_ratio'] = paired['vmean_v_rev'].abs() / paired['vmean_v_fwd']
paired['rpm_ratio'] = paired['rpm_rev'].abs() / paired['rpm_fwd']

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(paired['pwm_cmd'], paired['voltage_ratio'], 'o-', color='purple', label='|Vmean_rev| / Vmean_fwd')
ax.plot(paired['pwm_cmd'], paired['rpm_ratio'], 's-', color='green', label='|RPM_rev| / RPM_fwd')

ax.axhline(1.0, color='grey', linestyle='--', linewidth=1)
ax.set_title('Forward/Reverse Asymmetry Ratio vs PWM')
ax.set_xlabel('PWM command')
ax.set_ylabel('Asymmetry ratio')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('../figures/04_asymmetry_ratio.png', dpi=150)
plt.show()